In [ ]:
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
sns.set_theme(style='whitegrid')

interactions = pd.read_csv('../data/cleaned/interactions_clean.csv')
interactions['last_event'] = pd.to_datetime(interactions['last_event'])
print(interactions.shape)

In [ ]:
# Hold out each user's most recently interacted product
test = (
    interactions.sort_values('last_event')
    .groupby('user_id').tail(1)
    .set_index('user_id')['product_id']
    .to_dict()
)
test_user_ids = list(test.keys())

# Training set: everything except test items
is_test = interactions.apply(lambda r: test.get(r['user_id']) == r['product_id'], axis=1)
train   = interactions[~is_test].copy()

print(f'Train interactions: {len(train)}')
print(f'Test users:         {len(test_user_ids)}')

In [ ]:
def precision_at_k(recommended, relevant, k):
    """Fraction of top-K recommendations that are relevant."""
    return int(relevant in recommended[:k]) / k

def recall_at_k(recommended, relevant, k):
    """Fraction of relevant items retrieved in top-K."""
    return int(relevant in recommended[:k])  # 1 relevant item total

def ndcg_at_k(recommended, relevant, k):
    """NDCG@K (ideal DCG = 1 since there is one relevant item)."""
    for rank, item in enumerate(recommended[:k], start=1):
        if item == relevant:
            return 1.0 / np.log2(rank + 1)
    return 0.0

def evaluate(rec_fn, test_dict, k_values=(5, 10, 20)):
    results = {k: {'precision': [], 'recall': [], 'ndcg': []} for k in k_values}
    for user, held_out in test_dict.items():
        recs = rec_fn(user)  # returns list of product_ids
        for k in k_values:
            results[k]['precision'].append(precision_at_k(recs, held_out, k))
            results[k]['recall'].append(recall_at_k(recs, held_out, k))
            results[k]['ndcg'].append(ndcg_at_k(recs, held_out, k))
    return {
        k: {m: np.mean(v) for m, v in metrics.items()}
        for k, metrics in results.items()
    }

print('Metric helpers ready.')

In [ ]:
with open('../models/item_cf.pkl', 'rb') as f:
    item_cf = pickle.load(f)

ue_cf  = item_cf['user_enc']
ie_cf  = item_cf['item_enc']
R_cf   = item_cf['R']
sim    = item_cf['item_similarity']

def recommend_item_cf(user_id, n=50):
    if user_id not in ue_cf.classes_:
        return []
    u = ue_cf.transform([user_id])[0]
    user_vec = R_cf[u].toarray().flatten()
    scores   = user_vec @ sim
    # Mask already-seen items
    scores[user_vec > 0] = -1
    top_idx  = np.argsort(scores)[::-1][:n]
    return ie_cf.inverse_transform(top_idx).tolist()

cf_results = evaluate(recommend_item_cf, test)
print('Item-CF results:', cf_results)

In [ ]:
with open('../models/svd_model.pkl', 'rb') as f:
    svd_m = pickle.load(f)

U_svd = svd_m['U']
V_svd = svd_m['V']
ue_sv = svd_m['user_enc']
ie_sv = svd_m['item_enc']

def recommend_svd(user_id, n=50):
    if user_id not in ue_sv.classes_:
        return []
    u   = ue_sv.transform([user_id])[0]
    scores = U_svd[u] @ V_svd
    # Mask seen items (use original R from item_cf for the mask)
    seen = item_cf['R'][item_cf['user_enc'].transform([user_id])[0]].toarray().flatten() > 0
    scores[seen] = -1
    top_idx = np.argsort(scores)[::-1][:n]
    return ie_sv.inverse_transform(top_idx).tolist()

svd_results = evaluate(recommend_svd, test)
print('SVD results:', svd_results)

In [ ]:
class NCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32, hidden=[64, 32]):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        layers = []
        in_dim = emb_dim * 2
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1), nn.Sigmoid()]
        self.mlp = nn.Sequential(*layers)

    def forward(self, users, items):
        return self.mlp(torch.cat([self.user_emb(users), self.item_emb(items)], dim=1)).squeeze()

ckpt     = torch.load('../models/ncf_model.pt', map_location='cpu')
ue_nc    = ckpt['user_enc']
ie_nc    = ckpt['item_enc']
ncf_mdl  = NCF(ckpt['n_users'], ckpt['n_items'], ckpt['emb_dim'])
ncf_mdl.load_state_dict(ckpt['model_state'])
ncf_mdl.eval()

all_items = torch.arange(ckpt['n_items'])

def recommend_ncf(user_id, n=50):
    if user_id not in ue_nc.classes_:
        return []
    u     = int(ue_nc.transform([user_id])[0])
    utens = torch.tensor([u] * ckpt['n_items'])
    with torch.no_grad():
        scores = ncf_mdl(utens, all_items).numpy()
    # Mask seen items
    seen = item_cf['R'][item_cf['user_enc'].transform([user_id])[0]].toarray().flatten() > 0
    scores[seen] = -1
    top_idx = np.argsort(scores)[::-1][:n]
    return ie_nc.inverse_transform(top_idx).tolist()

ncf_results = evaluate(recommend_ncf, test)
print('NCF results:', ncf_results)

In [ ]:
K = 10
rows = []
for name, res in [('Item-CF', cf_results), ('SVD', svd_results), ('NCF', ncf_results)]:
    rows.append({'Model': name,
                 f'P@{K}':    round(res[K]['precision'], 4),
                 f'R@{K}':    round(res[K]['recall'], 4),
                 f'NDCG@{K}': round(res[K]['ndcg'], 4)})

results_df = pd.DataFrame(rows).set_index('Model')
print(results_df.to_string())

results_df.plot(kind='bar', figsize=(9, 4), rot=0,
                color=['#378ADD', '#1D9E75', '#EF9F27'])
plt.title(f'Model comparison at K={K}')
plt.ylabel('Score'); plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png'); plt.show()

In [ ]:
losses = ckpt['epoch_losses']
plt.figure(figsize=(7, 3))
plt.plot(range(1, len(losses)+1), losses, marker='o', color='#D85A30')
plt.title('NCF training loss'); plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.tight_layout()
plt.savefig('../data/processed/ncf_loss.png'); plt.show()

In [ ]:
ks = [5, 10, 20]
for name, res in [('Item-CF', cf_results), ('SVD', svd_results), ('NCF', ncf_results)]:
    ndcg_vals = [res[k]['ndcg'] for k in ks]
    plt.plot(ks, ndcg_vals, marker='o', label=name)

plt.title('NDCG@K comparison'); plt.xlabel('K'); plt.ylabel('NDCG')
plt.xticks(ks); plt.legend(); plt.tight_layout()
plt.savefig('../data/processed/ndcg_curve.png'); plt.show()